# XOR: Validação da MLP

Antes de aplicar a rede neural ao conjunto MNIST, a implementação será validada utilizando o problema XOR.

O XOR é um exemplo clássico utilizado para testar redes neurais porque seus dados não são linearmente separáveis. Isso significa que um perceptron simples não consegue resolver o problema corretamente, sendo necessária pelo menos uma camada oculta.

Como o conjunto possui apenas quatro exemplos, ele permite verificar com facilidade se as etapas de propagação direta (forward propagation), cálculo dos erros e retropropagação dos gradientes (backpropagation) foram implementadas corretamente.

Se a rede conseguir aprender o XOR e atingir uma boa taxa de acerto, haverá maior confiança de que a implementação está funcionando corretamente antes de avançar para um problema mais complexo como o MNIST.

## Etapa 1: Carregar XOR

In [1]:
import numpy as np

# 4 exemplos do XOR
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([[0], [1], [1], [0]])

print("X shape:", X.shape) 
print("y shape:", y.shape) 
print("\nDados:")
for i in range(len(X)):
    print(f"  {X[i]} → {y[i][0]}")

X shape: (4, 2)
y shape: (4, 1)

Dados:
  [0 0] → 0
  [0 1] → 1
  [1 0] → 1
  [1 1] → 0


## Etapa 2: Função de ativação (Sigmoid)

Para este experimento foi utilizada a função de ativação Sigmoid, que transforma qualquer valor real em um número entre 0 e 1. Essa característica é útil em problemas de classificação binária, como o XOR.

Fórmula: $σ(x) = \frac{1}{1 + e^{-x}}$


Uma vantagem da Sigmoid é que sua derivada pode ser calculada a partir do próprio valor da função, sem necessidade de operações adicionais complexas. Isso simplifica a implementação do algoritmo de backpropagation, utilizado para ajustar os pesos da rede durante o treinamento.

In [2]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

# Teste rápido
test_values = np.array([-2, -1, 0, 1, 2])
print("x:          ", test_values)
print("sigmoid(x): ", sigmoid(test_values).round(4))
print("derivada:   ", sigmoid_derivative(test_values).round(4))

x:           [-2 -1  0  1  2]
sigmoid(x):  [0.1192 0.2689 0.5    0.7311 0.8808]
derivada:    [0.105  0.1966 0.25   0.1966 0.105 ]


## Etapa 3: Inicialização dos pesos

Cada camada tem dois parâmetros: W (pesos) e b (bias).

- W conecta os neurônios entre camadas: `shape (neurônios_entrada, neurônios_saída)`
- b é somado depois da multiplicação: `shape (1, neurônios_saída)`

Inicializamos W com valores aleatórios pequenos diferentes de zero.
Se todos os pesos fossem iguais, todos os neurônios aprenderiam a mesma coisa ("problema de simetria"). Valores aleatórios quebram essa simetria.

Os bias foram inicializados com zero, pois o problema de simetria ocorre nos pesos das conexões entre neurônios. Como cada neurônio possui pesos diferentes, os bias podem começar em zero sem prejudicar o aprendizado.

In [3]:
np.random.seed(42)  # reprodutibilidade

# Camada 1: 2 entradas → 2 neurônios ocultos
W1 = np.random.randn(2, 2) * 0.1
b1 = np.zeros((1, 2))

# Camada 2: 2 neurônios ocultos → 1 saída
W2 = np.random.randn(2, 1) * 0.1
b2 = np.zeros((1, 1))

print("W1 shape:", W1.shape, "\n", W1)
print("\nb1 shape:", b1.shape, "\n", b1)
print("\nW2 shape:", W2.shape, "\n", W2)
print("\nb2 shape:", b2.shape, "\n", b2)

W1 shape: (2, 2) 
 [[ 0.04967142 -0.01382643]
 [ 0.06476885  0.15230299]]

b1 shape: (1, 2) 
 [[0. 0.]]

W2 shape: (2, 1) 
 [[-0.02341534]
 [-0.0234137 ]]

b2 shape: (1, 1) 
 [[0.]]


## Etapa 4: Forward Pass

O forward pass representa o fluxo de informações da entrada até a saída da rede neural.

Em cada camada são realizadas duas etapas:

1. Aplicação da transformação linear: z = XW + b
2. Aplicação da função de ativação: a = sigmoid(z)

Os valores z são chamados de pré-ativações, enquanto os valores a representam as ativações dos neurônios e são utilizados como entrada para a próxima camada.

Ao final do processo, a rede produz uma predição para cada exemplo do conjunto XOR.

In [5]:
def forward(X, W1, b1, W2, b2):
    # Camada 1
    z1 = X @ W1 + b1      # (4,2) @ (2,2) = (4,2)
    a1 = sigmoid(z1)       # (4,2)
    
    # Camada 2 (saída)
    z2 = a1 @ W2 + b2     # (4,2) @ (2,1) = (4,1)
    a2 = sigmoid(z2)       # (4,1)
    
    return z1, a1, z2, a2

z1, a1, z2, a2 = forward(X, W1, b1, W2, b2)

print("z1 shape:", z1.shape)
print("a1 shape:", a1.shape)
print("z2 shape:", z2.shape)
print("a2 shape:", a2.shape)
print("\nPredições iniciais:")
print(a2.round(4))
print("\nEsperado:")
print(y)

z1 shape: (4, 2)
a1 shape: (4, 2)
z2 shape: (4, 1)
a2 shape: (4, 1)

Predições iniciais:
[[0.4941]
 [0.4938]
 [0.4941]
 [0.4938]]

Esperado:
[[0]
 [1]
 [1]
 [0]]


## Etapa 5: Função de perda - Binary Cross-Entropy

Após realizar o forward pass, é necessário medir o quão distante a predição está do valor esperado. Para isso foi utilizada a função de perda Binary Cross-Entropy (BCE), adequada para problemas de classificação binária.

$$L=−n1​∑i=1n​[yi​log(y^​i​)+(1−yi​)log(1−y^​i​)]$$

A BCE penaliza previsões incorretas e produz valores menores quando a rede realiza previsões próximas dos valores reais. O objetivo do treinamento é ajustar os pesos da rede de forma a minimizar essa função de perda ao longo das épocas.

In [7]:
def binary_cross_entropy(y_true, y_pred):
    # clip evita log(0) que daria -infinito
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

loss_inicial = binary_cross_entropy(y, a2)
print("Loss inicial:", round(loss_inicial, 4))

Loss inicial: 0.6932


## Etapa 6: Backpropagation

Após calcular a função de perda, é necessário determinar como cada parâmetro da rede contribuiu para o erro obtido. Para isso é utilizado o algoritmo de Backpropagation.

O Backpropagation aplica a regra da cadeia do cálculo diferencial para propagar o erro da camada de saída até as camadas anteriores da rede. A partir desse processo são calculados os gradientes dos pesos e dos bias, que serão utilizados posteriormente para atualizar os parâmetros da rede.

Inicialmente são calculados os gradientes da camada de saída, relacionando a função de perda com a ativação final da rede. Em seguida, o erro é propagado para a camada oculta, permitindo calcular os gradientes dos pesos e bias dessa camada.

Os gradientes obtidos indicam a direção em que cada parâmetro deve ser ajustado para reduzir a função de perda durante o treinamento.

Gradientes da camada de saída:
  - dL/da2 = -(y/a2 - (1-y)/(1-a2))   ← derivada da loss
  - dL/dz2 = dL/da2 · sigmoid'(z2)    ← passa pela ativação
  - dL/dW2 = a1.T @ dL/dz2            ← gradiente dos pesos
  - dL/db2 = soma de dL/dz2           ← gradiente do bias

Gradientes da camada oculta:
  - dL/da1 = dL/dz2 @ W2.T             ← propaga o erro pra trás
  - dL/dz1 = dL/da1 · sigmoid'(z1)     ← passa pela ativação
  - dL/dW1 = X.T @ dL/dz1              ← gradiente dos pesos
  - dL/db1 = soma de dL/dz1            ← gradiente do bias

In [8]:
def backward(X, y, z1, a1, z2, a2, W2):
    n = X.shape[0]  # número de exemplos (4)
    
    # Camada de saída
    dL_da2 = -(y / a2 - (1 - y) / (1 - a2))
    dL_dz2 = dL_da2 * sigmoid_derivative(z2)
    dL_dW2 = a1.T @ dL_dz2 / n
    dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True) / n
    
    # Camada oculta
    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * sigmoid_derivative(z1)
    dL_dW1 = X.T @ dL_dz1 / n
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True) / n
    
    return dL_dW1, dL_db1, dL_dW2, dL_db2

dL_dW1, dL_db1, dL_dW2, dL_db2 = backward(X, y, z1, a1, z2, a2, W2)

print("Gradiente W1:\n", dL_dW1.round(6))
print("\nGradiente W2:\n", dL_dW2.round(6))

Gradiente W1:
 [[2.0e-05 2.1e-05]
 [2.0e-05 1.7e-05]]

Gradiente W2:
 [[-0.00311 ]
 [-0.003124]]


## Etapa 7: Atualização dos pesos

Após calcular os gradientes através do Backpropagation, os parâmetros da rede são atualizados utilizando Gradient Descent.

A atualização consiste em mover cada peso na direção oposta ao gradiente, reduzindo gradualmente a função de perda.

A regra de atualização é dada por: $W=W−η\frac{∂W}{∂L}​$ onde η representa o learning rate. Esse hiperparâmetro controla o tamanho do passo dado durante o processo de otimização.

Valores muito altos podem fazer a função de perda oscilar sem convergir, enquanto valores muito baixos tornam o treinamento excessivamente lento.

In [9]:
def update_weights(W1, b1, W2, b2, dL_dW1, dL_db1, dL_dW2, dL_db2, lr=0.1):
    W1 = W1 - lr * dL_dW1
    b1 = b1 - lr * dL_db1
    W2 = W2 - lr * dL_dW2
    b2 = b2 - lr * dL_db2
    return W1, b1, W2, b2

## Etapa 8: Loop de treinamento

Uma época = passar todos os exemplos uma vez pela rede.
A cada época: forward pass → calcula loss → backward → atualiza pesos.
Repetimos por muitas épocas até a loss convergir.

In [11]:
# Reinicia os pesos do zero
np.random.seed(42)
W1 = np.random.randn(2, 2) * 0.1
b1 = np.zeros((1, 2))
W2 = np.random.randn(2, 1) * 0.1
b2 = np.zeros((1, 1))

lr = 0.1
epochs = 10000
loss_history = []

for epoch in range(epochs):
    # Forward
    z1, a1, z2, a2 = forward(X, W1, b1, W2, b2)
    
    # Loss
    loss = binary_cross_entropy(y, a2)
    loss_history.append(loss)
    
    # Backward
    dL_dW1, dL_db1, dL_dW2, dL_db2 = backward(X, y, z1, a1, z2, a2, W2)
    
    # Atualiza
    W1, b1, W2, b2 = update_weights(W1, b1, W2, b2, dL_dW1, dL_db1, dL_dW2, dL_db2, lr)
    
    if epoch % 1000 == 0:
        print(f"Época {epoch:5d} | Loss: {loss:.4f}")

print(f"\nLoss final: {loss:.4f}")
print("\nPredições finais:")
print(a2.round(3))
print("\nEsperado:")
print(y)

Época     0 | Loss: 0.6932
Época  1000 | Loss: 0.6931
Época  2000 | Loss: 0.6931
Época  3000 | Loss: 0.6931
Época  4000 | Loss: 0.6931
Época  5000 | Loss: 0.6931
Época  6000 | Loss: 0.6931
Época  7000 | Loss: 0.6931
Época  8000 | Loss: 0.6931
Época  9000 | Loss: 0.6931

Loss final: 0.6931

Predições finais:
[[0.5]
 [0.5]
 [0.5]
 [0.5]]

Esperado:
[[0]
 [1]
 [1]
 [0]]
